# LSNM-UV-X Experiments
Run Cells 1–2 once to set up, then either the **test run** (Cell 3a) or the **full run** (Cell 3b).

In [ ]:
# Cell 1 — Setup
%matplotlib inline
import logging
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
# Suppress noisy output from experiment runner; warnings still shown
logging.basicConfig(level=logging.WARNING)

In [ ]:
%%capture
# Cell 2 — Imports (captured to suppress lingam/causallearn startup messages)
from run_experiments import (
    run_all_experiments,
    run_alpha_sensitivity,
    plot_results,
    plot_bidir_results,
    plot_runtime,
    plot_alpha_sensitivity,
)
# Force-reload in case you edited the source files between runs
import importlib
import run_experiments, lsnm_data_gen, lsnm_uv_x, eval_metrics
for m in [eval_metrics, lsnm_data_gen, lsnm_uv_x, run_experiments]:
    importlib.reload(m)
from run_experiments import (
    run_all_experiments, run_alpha_sensitivity,
    plot_results, plot_bidir_results, plot_runtime, plot_alpha_sensitivity,
)

In [ ]:
# Cell 3a — TEST RUN (3 sample sizes x 5 trials, ~2 min without BANG, ~5 min with)
# Run this first to verify the pipeline works end-to-end.
df_main = run_all_experiments(
    n_list       = [100, 500, 1000],
    n_trials     = 5,
    alpha        = 0.01,
    d            = 3,
    include_bang = True,
    n_jobs       = 1,        # sequential — avoids rpy2 fork issues with BANG
    save_path    = 'results_test.csv',
)

In [ ]:
# Cell 3b — FULL RUN (10 sample sizes x 100 trials, ~30-90 min)
# Only run after Cell 3a confirms the pipeline is working.
df_main = run_all_experiments(
    n_list       = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000],
    n_trials     = 100,
    alpha        = 0.01,
    d            = 3,
    include_bang = True,
    n_jobs       = 1,        # sequential — avoids rpy2 fork issues with BANG
    save_path    = 'results_section6.csv',
)

In [ ]:
# Cell 4 — Sanity check: confirm run completed and show mean F1 per method/n
print(f"Trials completed : {len(df_main)}")
print(f"Methods          : {df_main['method'].unique().tolist()}")
print(f"Sample sizes     : {sorted(df_main['n'].unique())}")
print()
print(df_main.groupby(['method', 'n'])[['f1_directed', 'f1_bidir']].mean().round(3).to_string())

In [ ]:
# Cell 5 — Directed edge performance (Precision / Recall / F1 vs n, all methods)
plot_results(df_main, save_path='figure_section6_directed.png')

In [ ]:
# Cell 6 — Bidirected (UBP/UCP) performance (Precision / Recall / F1 vs n, all methods)
plot_bidir_results(df_main, save_path='figure_section6_bidir.png')

In [ ]:
# Cell 7 — Runtime vs sample size
plot_runtime(df_main, save_path='figure_runtime.png')

In [ ]:
# Cell 8 — Alpha sensitivity (LSNM-UV-X only, n=500, 100 trials per alpha)
df_alpha = run_alpha_sensitivity(
    alpha_list = [0.5, 0.1, 0.05, 0.01, 0.005, 0.001],
    n          = 500,
    n_trials   = 100,
    save_path  = 'results_alpha_sensitivity.csv',
)

In [ ]:
# Cell 9 — Alpha sensitivity plot
plot_alpha_sensitivity(df_alpha, save_path='figure_alpha_sensitivity.png')